# Thêm Thư Viện

In [1]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [2]:
conn_libol = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=libol;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2025;'
)
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=Library_DWH;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2025;'
)

## Đọc data từ SQL Server

In [3]:
# Đọc dữ liệu từ bảng Lop trong CSDL libol
query_Dangtailieu = """SELECT Dang_tai_lieu_ID, 
                        dbo.DecodeUTF8String(Dang_tai_lieu) AS Dang_tai_lieu, 
                        dbo.DecodeUTF8String(Ky_hieu_tai_lieu) AS Ky_hieu_tai_lieu,
                        LoanPeriod,
                        Renewals,
                        RenewalPeriod,
                        TimeUnit,
                        Fee,
                        OverdueFine,
                        FixedFee
                        FROM Dang_tai_lieu
                        """
df_dangtailieu = pd.read_sql(query_Dangtailieu, conn_libol)
print(df_dangtailieu)

    Dang_tai_lieu_ID                                    Dang_tai_lieu  \
0                  1                     Sách, chuyên khảo, tuyển tập   
1                  2                                        Bài trích   
2                  3                                Luận án, luận văn   
3                  4  Báo cáo kết quả  nghiên cứu; tổng kết; khảo sát   
4                  5                                 Báo cáo hội nghị   
5                  6                               Catalô công nghiệp   
6                  7                                       Tiêu chuẩn   
7                  8                                         Sáng chế   
8                  9                                  ấn phẩm định kỳ   
9                 10                                             Phim   
10                11                              Bản đồ, sách bản đồ   
11                12                    Hình vẽ, bản vẽ, tranh ảnh,..   
12                13                     Tờ rời giớ

C:\Users\admin\AppData\Local\Temp\ipykernel_30008\3235927709.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dangtailieu = pd.read_sql(query_Dangtailieu, conn_libol)


## Xử lý data

In [4]:
new_row = pd.DataFrame({'Dang_tai_lieu_ID': [0],
                        'Dang_tai_lieu': ['(Không xác định)'],
                        'Ky_hieu_tai_lieu': ['None'],
                        'Fee': [0],
                        'OverdueFine': [0],
                        'FixedFee': ['False'],
                        }) # Tạo hàng dữ liệu giả lập cho nhóm không xác định
df_dangtailieu = pd.concat([df_dangtailieu, new_row], ignore_index=True) # Thêm vào dataframe

columns_to_check = ['LoanPeriod', 'Renewals', 'RenewalPeriod', 'TimeUnit']
for index, row in df_dangtailieu.iterrows():
    for col in columns_to_check:
        if pd.isna(row[col]):
            df_dangtailieu.at[index, col] = 0  # Thay NaN bằng 0

df_dangtailieu = df_dangtailieu.sort_values(by="Dang_tai_lieu_ID", ascending=True).reset_index(drop=True) # sắp xếp từ nhỏ đến lớn           
print(df_dangtailieu)

    Dang_tai_lieu_ID                                    Dang_tai_lieu  \
0                  0                                 (Không xác định)   
1                  1                     Sách, chuyên khảo, tuyển tập   
2                  2                                        Bài trích   
3                  3                                Luận án, luận văn   
4                  4  Báo cáo kết quả  nghiên cứu; tổng kết; khảo sát   
5                  5                                 Báo cáo hội nghị   
6                  6                               Catalô công nghiệp   
7                  7                                       Tiêu chuẩn   
8                  8                                         Sáng chế   
9                  9                                  ấn phẩm định kỳ   
10                10                                             Phim   
11                11                              Bản đồ, sách bản đồ   
12                12                    Hình vẽ, bả

## Load data

### [Nếu cần] Clear bảng

In [6]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM olap.DIM_Dang_tai_lieu"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

### Load data vào bảng Dim

In [7]:
cursor_dwh = conn_dwh_library.cursor()
insert_query = """
                INSERT INTO olap.DIM_Dang_tai_lieu (ID_dang_tai_lieu, Dang_tai_lieu, Ky_hieu_tai_lieu, LoanPeriod, Renewals, RenewalPeriod, TimeUnit, Fee, OverdueFine, FixedFee) 
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
               """
for index, row in df_dangtailieu.iterrows():
    values = (row['Dang_tai_lieu_ID'], 
                row['Dang_tai_lieu'],
                row['Ky_hieu_tai_lieu'],
                row['LoanPeriod'],
                row['Renewals'],
                row['RenewalPeriod'],
                row['TimeUnit'],
                row['Fee'],
                row['OverdueFine'],
                row['FixedFee'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_library.commit()